In [ ]:
# 📦 初回のみ実行
!pip install bert-score rouge-score nltk

# 📚 ライブラリ読み込み
import pandas as pd
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# 📂 CSV読み込み
df = pd.read_csv("data/evaluation_dataset.csv", encoding="utf-8")

# 🧹 列名と中身を整える
df = df.rename(columns={
    "Question": "question",
    "Answe_generaltive AI": "reference",
    "Answe_Local LLM-RAG_e": "candidate"
})

df["question"] = df["question"].str.replace(r'^Q:\s*', '', regex=True)
df["reference"] = df["reference"].str.replace(r'^A:\s*', '', regex=True)
df["candidate"] = df["candidate"].str.replace(r'^A:\s*', '', regex=True)

df = df.dropna(subset=["reference", "candidate"])

# ✅ BERTScore（英語）
P, R, F1 = bert_score(df["candidate"].tolist(), df["reference"].tolist(), lang="en", verbose=True)

# ✅ ROUGEとBLEUの計算
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1
rouge_scores = []
bleu_scores = []

for ref, cand in zip(df["reference"], df["candidate"]):
    rouge_f = rouge.score(ref, cand)['rougeL'].fmeasure
    bleu = sentence_bleu([ref.split()], cand.split(), smoothing_function=smooth)
    rouge_scores.append(rouge_f)
    bleu_scores.append(bleu)

# 📊 結果を追加
df["BERTScore(F1)"] = F1.tolist()
df["ROUGE-L"] = rouge_scores
df["BLEU"] = bleu_scores
df["difficulty"] = df["question"].apply(len)

# ✅ 保存と表示
df.to_csv("evaluation_results_en.csv", index=False)
from IPython.display import display
display(df)

In [ ]:
# 📦 初回だけ実行
!pip install bert-score rouge-score nltk

# 📚 ライブラリ読み込み
import pandas as pd
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# 📂 CSV読み込み（文字化け対策：Shift_JIS）
df = pd.read_csv("evaluation_dataset.csv", encoding="utf-8")

# 🧹 列名と中身を整える
df = df.rename(columns={
    "Question (JPN)": "question",
    "Reference (JPN)": "reference",
    "Gemma-3 (JPN)": "candidate"
})

df["question"] = df["question"].str.replace(r'^Q:\s*', '', regex=True)
df["reference"] = df["reference"].str.replace(r'^A:\s*', '', regex=True)
df["candidate"] = df["candidate"].str.replace(r'^A:\s*', '', regex=True)

df = df.dropna(subset=["reference", "candidate"])

# ✅ BERTScore（英語）
P, R, F1 = bert_score(df["candidate"].tolist(), df["reference"].tolist(), lang="en", verbose=True)

# ✅ ROUGEとBLEUの計算
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1
rouge_scores = []
bleu_scores = []

for ref, cand in zip(df["reference"], df["candidate"]):
    rouge_f = rouge.score(ref, cand)['rougeL'].fmeasure
    bleu = sentence_bleu([ref.split()], cand.split(), smoothing_function=smooth)
    rouge_scores.append(rouge_f)
    bleu_scores.append(bleu)

# 📊 結果を追加
df["BERTScore(F1)"] = F1.tolist()
df["ROUGE-L"] = rouge_scores
df["BLEU"] = bleu_scores
df["difficulty"] = df["question"].apply(len)

# ✅ 結果保存と表示
df.to_csv("evaluation_results_en.csv", index=False, encoding="shift_jis")
from IPython.display import display
display(df)


In [ ]:
df